<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "174"
iso_country = "Comoros"
year = 2021
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.270*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Comoros': 

Number of trade records present in dataset: 6709
Number of partner countries present in dataset: 80
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$314.8m
Total weight within the dataset: 752.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,174,Comoros,784,United Arab Emirates,Import,Sea,551449,"551449 | Fabrics, woven; prin",5514,5514 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,360.441,NaN,360.441,kg,150.0,150.0,64.735204,5540.0,3442.395340,222.844163,['Container'],['Container'],1.164768,1.162735,1.162933,1.162933,1.163845,1.163845,1.218407,2.159399,1.343942,1.247725,1.251114,1.282449,1.193034,2.114801,1.414584,1.295737,1.266003,1.323340,1.170781,2.299703,1.754152,1.534119,1.419351,1.379312,1.148941,2.445148,2.012076,1.885486,1.519597,1.478278,1.127512,2.477815,2.367333,2.387687,1.647401,1.552883,1.105654,2.426886,2.325700,2.548146,1.884846,1.581345,1.085063,2.164841,2.256341,2.364409,1.772436,1.600029,1.064065,1.988591,2.021455,2.521504,2.083117,1.610625,2676
1,1,174,Comoros,450,Madagascar,Import,Sea,560121,560121 | Wadding; other artic,5601,5601 | Wadding of textile m,56,"56 | Wadding, felt and no",27991,Wadding of textile materials and articles ther...,Textiles and Wearing Apparel,27,TEX - Textiles,42.324,NaN,42.324,kg,16.0,16.0,6.162374,1347.0,836.986737,5.157826,['Container'],['Container'],0.026959,0.026912,0.026917,0.026917,0.026938,0.026938,0.028201,0.049980,0.031106,0.028879,0.028958,0.029683,0.027613,0.048948,0.032741,0.029990,0.029302,0.030629,0.027098,0.053228,0.040601,0.035508,0.032851,0.031925,0.026593,0.056594,0.046570,0.043640,0.035172,0.034215,0.026097,0.057350,0.054793,0.055264,0.038130,0.035942,0.025591,0.056171,0.053829,0.058978,0.043626,0.036601,0.025114,0.050106,0.052224,0.054725,0.041024,0.037033,0.024628,0.046027,0.046787,0.058361,0.048215,0.037279,27991


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $82.0m (26.0%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
10 | Cereals               33884458.1  58861065.1     10.8      7.8
04 | Dairy produce; birds   6957953.2   3644786.3      2.2      0.5
11 | Products of the mill   6840767.5  16525508.1      2.2      2.2
22 | Beverages, spirits a   5775574.4  39576720.0      1.8      5.3
17 | Sugars and sugar con   5440932.6  10459801.6      1.7      1.4
02 | Meat and edible meat   5066398.4   2428698.6      1.6      0.3
20 | Preparations of vege   4468235.6   3688610.0      1.4      0.5
15 | Animal, vegetable or   4365846.5   2957472.7      1.4      0.4
19 | Preparations of cere   2603806.3   1048943.3      0.8      0.1
16 | Meat, fish, crustace   2288213.9    484963.2      0.7      0.1
07 | Vegetables and certa   1126106.3   1940797.9      0.4      0.3
21 | Miscellaneo

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $77.6m (24.6%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       58663388.5  86155438.9     18.6     11.4
Food & Beverages  18890322.4  47998769.1      6.0      6.4
Fishing              45128.7      6280.8      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $71.6m (22.7%), ... 

                                                cifvalue  vol_kg_vtw  pct_val  \
gtap_desc                                                                       
PCR - Processed rice                          33882232.3  58853437.5     10.8   
OFD - Food products n.e.c.                    18825330.6  22457006.5      6.0   
MIL - Dairy products                           6865266.4   3599943.7      2.2   
VOL - Vegetable oils and fats                  4364478.1   2957158.7      1.4   
OMT - Meat products n.e.c.                     4196486.7   2235462.7      1.3   
SGR - Sugar                                    1157877.8   2539614.3      0.4   
V_F - Vegetables, fruit, nuts                  1155965.2   1626457.5      0.4   
CMT - Bovine meat prods                         931066.5    207813.7      0.3   
OCR - Crops n.e.c.                              113932.6     74045.7      0.0   
CTL - Bovine cattle, sheep and goats, horses

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 26.0% 
GTAP 22.7% 
EORA 24.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46353/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,174,Comoros,784,United Arab Emirates,Import,Sea,551449,"551449 | Fabrics, woven; prin",5514,5514 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,360.441,NaN,360.441,kg,150.0,150.0,64.735204,5540.0,3442.395340,222.844163,['Container'],['Container'],1.164768,1.162735,1.162933,1.162933,1.163845,1.163845,1.218407,2.159399,1.343942,1.247725,1.251114,1.282449,1.193034,2.114801,1.414584,1.295737,1.266003,1.323340,1.170781,2.299703,1.754152,1.534119,1.419351,1.379312,1.148941,2.445148,2.012076,1.885486,1.519597,1.478278,1.127512,2.477815,2.367333,2.387687,1.647401,1.552883,1.105654,2.426886,2.325700,2.548146,1.884846,1.581345,1.085063,2.164841,2.256341,2.364409,1.772436,1.600029,1.064065,1.988591,2.021455,2.521504,2.083117,1.610625,2676,NaN
1,1,174,Comoros,450,Madagascar,Import,Sea,560121,560121 | Wadding; other artic,5601,5601 | Wadding of textile m,56,"56 | Wadding, felt and no",27991,Wadding of textile materials and articles ther...,Textiles and Wearing Apparel,27,TEX - Textiles,42.324,NaN,42.324,kg,16.0,16.0,6.162374,1347.0,836.986737,5.157826,['Container'],['Container'],0.026959,0.026912,0.026917,0.026917,0.026938,0.026938,0.028201,0.049980,0.031106,0.028879,0.028958,0.029683,0.027613,0.048948,0.032741,0.029990,0.029302,0.030629,0.027098,0.053228,0.040601,0.035508,0.032851,0.031925,0.026593,0.056594,0.046570,0.043640,0.035172,0.034215,0.026097,0.057350,0.054793,0.055264,0.038130,0.035942,0.025591,0.056171,0.053829,0.058978,0.043626,0.036601,0.025114,0.050106,0.052224,0.054725,0.041024,0.037033,0.024628,0.046027,0.046787,0.058361,0.048215,0.037279,27991,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $68.5m of $314.8m (21.8%)
Volume: 99.7 of 752.8 million kilograms (13.2%)
Value-to-Weight: 0.69 from 0.42 US$ per kg (+64.3%)
Average distance to market (volume-weighted): 3504.7 from 3852.3 nm (+-9.0%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    205  44574415.1  76943355.4     14.2     10.2
1                Dairy     94   6863604.1   3597754.0      2.2      0.5
2      Sugar and Honey     74   5442500.5  10460251.1      1.7      1.4
3        Fats and Oils     50   4138496.1   2691121.6      1.3      0.4
4        Meat and Prep     20   4135791.9   2220951.9      1.3      0.3
5           Vegetables    105   3071016.4   3556543.9      1.0      0.5
6               Fruits     42    303708.0    233002.9      0.1      0.0
7  Unprocessed Cereals      5      2225.8      7627.6      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,8491934.7,14999946.8,9354418.1,8690454.2,8714147.8,8933742.2,7863026.5,17233584.5,16462739.7,16617346.3,11465801.2,10804718.2,7423749.8,13831973.9,14064496.7,17490379.0,14454748.3,11204103.0
1,Fats and Oils,2719939.0,4647946.0,2953194.9,2767746.5,2776252.6,2846216.3,2531283.5,5558371.3,5265271.3,5359134.4,3649883.1,3418009.4,2397437.8,4327110.8,4419226.9,5239385.8,4328213.7,3538538.1
2,Fruits,463220.6,820967.7,510946.1,474366.4,475654.7,487568.0,428664.2,942025.6,900022.0,907761.6,626316.6,590381.9,404542.9,756031.1,768525.7,958632.0,791965.0,612334.2
3,Meat and Prep,9059549.8,16056364.6,9992973.8,9277546.3,9302742.0,9535737.1,8383696.2,18423963.1,17602466.4,17753813.8,12249368.4,11546570.5,7911933.1,14786310.3,15030671.2,18748822.5,15489158.1,11975914.1
4,Processed Cereals,68322098.5,112991918.1,73156511.6,68705793.5,68968712.5,71162242.1,63941416.0,131928109.1,126103569.9,129505509.0,90346027.6,83669603.2,60883419.1,107983896.0,111225612.4,129291661.6,108367574.8,86159481.0
5,Sugar and Honey,13178230.8,22934794.3,14422101.7,13430926.1,13471141.0,13831924.2,12232472.7,26437824.9,25260831.9,25597774.6,17710188.1,16620113.3,11572148.7,21321655.5,21746716.7,26638930.3,22085924.7,17206649.6
6,Unprocessed Cereals,2898.6,4953.8,3147.3,2939.8,2949.5,3033.9,2698.6,5735.7,5481.3,5581.0,3873.2,3617.1,2559.0,4651.1,4761.0,5724.6,4764.6,3737.5
7,Vegetables,2870850.5,4819992.8,3100786.1,2908739.7,2917070.9,2997678.2,2681274.1,5632756.9,5372225.1,5487579.6,3793508.2,3559726.1,2547545.5,4538432.3,4633673.8,5504868.5,4577246.5,3679311.3


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.95,0.13,0.03,0.03,0.06,1.37,1.25,1.28,0.52,0.43,0.93,0.97,1.47,1.02,0.55
1,Fats and Oils,0.47,0.06,0.01,0.01,0.03,0.73,0.66,0.68,0.27,0.21,0.47,0.49,0.69,0.47,0.28
2,Fruits,1.18,0.16,0.04,0.04,0.08,1.69,1.55,1.58,0.65,0.53,1.16,1.20,1.82,1.28,0.68
3,Meat and Prep,1.69,0.23,0.05,0.06,0.12,2.43,2.23,2.27,0.93,0.76,1.66,1.72,2.62,1.83,0.98
4,Processed Cereals,1.00,0.11,0.01,0.01,0.06,1.53,1.39,1.47,0.59,0.44,1.06,1.13,1.53,1.07,0.57
5,Sugar and Honey,1.79,0.23,0.05,0.05,0.12,2.61,2.39,2.46,1.01,0.81,1.79,1.87,2.77,1.93,1.04
6,Unprocessed Cereals,0.92,0.11,0.02,0.02,0.06,1.36,1.25,1.30,0.53,0.41,0.94,0.99,1.42,0.99,0.53
7,Vegetables,0.63,0.07,0.01,0.02,0.04,0.96,0.88,0.91,0.36,0.29,0.65,0.68,0.96,0.66,0.37
